# Ormophine · MySQL — تست‌های `if_ / else_` (خودکفا)

تمام fixture ها و تست‌ها در خود نوتبوک تعریف شده‌اند.

**پیش‌نیازها:**
1. پکیج `Ormophine` قابل import باشد (نصب یا از پوشه‌ی پروژه).
2. MySQL در حال اجرا باشد.

سلول ۱ تا ۵ را به ترتیب اجرا کن، سپس سلول ۶ (اجرای تست‌ها).

In [ ]:
# 1) نصب pytest و تنظیم متغیرهای محیطی MySQL
!pip install pytest --quiet

import os
os.environ["MYSQL_HOST"]     = "localhost"
os.environ["MYSQL_PORT"]     = "3306"
os.environ["MYSQL_USER"]     = "root"
os.environ["MYSQL_PASSWORD"] = ""            # اگر رمز داری اینجا بگذار
os.environ["MYSQL_DB_NAME"]  = "test_orm_db_fixed"

# اگر Ormophine در sys.path نیست، مسیر را باز کن:
# import sys
# sys.path.insert(0, "/path/to/folder_containing_Ormophine")

print("ENV set:")
for k in ("MYSQL_HOST", "MYSQL_PORT", "MYSQL_USER", "MYSQL_DB_NAME"):
    print(f"  {k} = {os.environ[k]}")

In [ ]:
# 2) import ها و بررسی اینکه LiteralValue در MySQL export شده
import pytest
import uuid
import os
from Ormophine import Mysql

print("Ormophine.Mysql file:", Mysql.__file__)
print("LiteralValue exported:", hasattr(Mysql, "LiteralValue"))
assert hasattr(Mysql, "LiteralValue"), (
    "LiteralValue در بکند MySQL export نشده. "
    "خط `from .Core.columnsoperation import ... LiteralValue` را به __init__.py اضافه کن."
)

# یادآوری برای دسترسی سریع در سلول‌های بعدی
MYSQL_HOST     = os.environ["MYSQL_HOST"]
MYSQL_PORT     = int(os.environ["MYSQL_PORT"])
MYSQL_USER     = os.environ["MYSQL_USER"]
MYSQL_PASSWORD = os.environ["MYSQL_PASSWORD"]
MYSQL_DB_NAME  = os.environ["MYSQL_DB_NAME"]

In [ ]:
# 3) fixture ها (scope=module برای driver اصلی + fixture های مستقل هر تست)

@pytest.fixture(scope="module")
def in_driver():
    """یک driver برای کل ماژول؛ DB را در صورت نیاز می‌سازد."""
    try:
        drv = Mysql.Driver(
            host=MYSQL_HOST, port=MYSQL_PORT, username=MYSQL_USER,
            password=MYSQL_PASSWORD, db_name=MYSQL_DB_NAME, create_new_db=True,
            charset="utf8mb4", collate="utf8mb4_bin"
        )
    except Exception:
        drv = Mysql.Driver(
            host=MYSQL_HOST, port=MYSQL_PORT, username=MYSQL_USER,
            password=MYSQL_PASSWORD, db_name=MYSQL_DB_NAME,
            charset="utf8mb4", collate="utf8mb4_bin"
        )
    yield drv
    try:
        drv.disconnect()
    except Exception:
        pass


@pytest.fixture
def users_if(in_driver):
    """جدول تازه با (id, name, age, active) برای تست‌های conditional.
    active در MySQL به‌صورت TINYINT(1) ذخیره می‌شود، پس هنگام خواندن 1/0 برمی‌گردد."""
    name = f"users_if_{uuid.uuid4().hex[:8]}"
    s = Mysql.TableStructure(name)
    s.add_column("id",     Mysql.DataTypes.INT(), primary_key=True)
    s.add_column("name",   Mysql.DataTypes.VARCHAR(100))
    s.add_column("age",    Mysql.DataTypes.INT())
    s.add_column("active", Mysql.DataTypes.BOOLEAN())
    in_driver.create_table(s)
    tbl = getattr(in_driver, name)
    tbl.bulk_insert(
        [tbl.id, tbl.name, tbl.age, tbl.active],
        [
            (1, 'Ali',   30,   1),
            (2, 'Reza',  17,   1),
            (3, 'Sara',  25,   0),
            (4, None,    40,   1),
            (5, 'Nima',  None, 0),
        ]
    )
    yield tbl
    try:
        in_driver.delete_table(tbl, True, True, True)
    except Exception:
        pass

In [ ]:
# 4) تست‌ها — SQL generation و functional و error handling

# ------------------------------------------------------ SQL generation

def test_if_sql_uses_if_function(users_if):
    """SQL باید `IF(cond, then, else)` (تابع MySQL) تولید کند،
    نه IIF (SQLite) و نه CASE WHEN (PostgreSQL)."""
    expr = users_if.name.if_(users_if.active == True).else_('inactive')
    sql, params = expr._output
    assert sql.startswith('(IF(')
    assert sql.endswith('))')
    assert 'IIF' not in sql.upper()
    assert 'CASE WHEN' not in sql.upper()
    assert params == [True, 'inactive']
    assert users_if.name.name in sql
    assert users_if.active.name in sql


def test_if_sql_then_literal_else_column(users_if):
    expr = (
        Mysql.LiteralValue('n/a')
        .if_(users_if.age == None)
        .else_(users_if.age)
    )
    sql, params = expr._output
    assert sql == (
        f'(IF(({users_if.age.name} IS NULL), %s, {users_if.age.name}))'
    )
    assert params == ['n/a']


def test_if_sql_then_and_else_both_columns(users_if):
    expr = users_if.name.if_(users_if.active == True).else_(users_if.name)
    sql, params = expr._output
    assert sql == (
        f'(IF(({users_if.active.name} = %s), '
        f'{users_if.name.name}, {users_if.name.name}))'
    )
    assert params == [True]


def test_if_sql_then_and_else_both_operations(users_if):
    expr = (
        users_if.name.upper()
        .if_(users_if.active == True)
        .else_(users_if.name.lower())
    )
    sql, params = expr._output
    assert f'(UPPER({users_if.name.name}))' in sql
    assert f'(LOWER({users_if.name.name}))' in sql
    assert params == [True]


# ------------------------------------------------------ functional

def test_if_01_column_then_literal_else(users_if):
    res = users_if.get_row(
        [users_if.name.if_(users_if.active == True).else_('inactive')],
        order_by=users_if.id,
    )
    assert res == ['Ali', 'Reza', 'inactive', None, 'inactive']


def test_if_02_literal_then_column_else(users_if):
    res = users_if.get_row(
        [Mysql.LiteralValue('n/a').if_(users_if.age == None).else_(users_if.age)],
        order_by=users_if.id,
    )
    assert res == [30, 17, 25, 40, 'n/a']


def test_if_03_raw_condition_auto_wrapped(users_if):
    res_true = users_if.get_row(
        [users_if.name.if_(True).else_('never')],
        order_by=users_if.id,
    )
    assert res_true == ['Ali', 'Reza', 'Sara', None, 'Nima']

    res_false = users_if.get_row(
        [users_if.name.if_(False).else_('always')],
        order_by=users_if.id,
    )
    assert res_false == ['always', 'always', 'always', 'always', 'always']


def test_if_04_expression_branches(users_if):
    expr = (
        users_if.name.upper()
        .if_(users_if.active == True)
        .else_(users_if.name.lower())
    )
    res = users_if.get_row([expr], order_by=users_if.id)
    assert res == ['ALI', 'REZA', 'sara', None, 'nima']


def test_if_05_nested_conditionals(users_if):
    label = (
        users_if.name
        .if_(users_if.age == None).else_('has_age')
        .if_(users_if.active == False).else_('active')
    )
    res = users_if.get_row([label], order_by=users_if.id)
    assert res == ['active', 'active', 'has_age', 'active', 'Nima']


# ---------------------------------------------------- error handling

def test_if_06_forgot_else_raises_runtime_error(users_if):
    bad = users_if.name.if_(users_if.active == True)
    with pytest.raises(RuntimeError, match="never chained"):
        users_if.get_row([bad])


def test_if_07_else_on_column_raises(users_if):
    with pytest.raises(RuntimeError, match="must be chained after"):
        users_if.name.else_('x')


def test_if_08_else_on_columns_operation_raises(users_if):
    op = users_if.name.upper()
    with pytest.raises(RuntimeError, match="must be chained after"):
        op.else_('x')


def test_if_09_partial_builder_in_where_raises(users_if):
    bad = users_if.age.if_(users_if.age > 18)
    with pytest.raises(RuntimeError, match="never chained"):
        users_if.get_row([users_if.id], where=bad)


# ---------------------------------------- keyword form and mixing

def test_if_10_keyword_form(users_if):
    res = users_if.get_row(
        [users_if.name.if(users_if.active == True).else('inactive')],
        order_by=users_if.id,
    )
    assert res == ['Ali', 'Reza', 'inactive', None, 'inactive']


def test_if_11_mixed_keyword_and_underscore_form(users_if):
    a = users_if.name.if_(users_if.active == True).else('X')
    b = users_if.name.if(users_if.active == True).else_('Y')
    res = users_if.get_row([a, b], order_by=users_if.id)
    assert res == (
        ('Ali', 'Ali'),
        ('Reza', 'Reza'),
        ('X', 'Y'),
        (None, None),
        ('X', 'Y'),
    )


# ------------------------------------ compound conditions

def test_if_12_compound_and(users_if):
    cond = (users_if.active == True) & (users_if.age > 18)
    res = users_if.get_row(
        [users_if.name.if_(cond).else_('nope')],
        order_by=users_if.id,
    )
    assert res == ['Ali', 'nope', 'nope', None, 'nope']


def test_if_13_or_condition(users_if):
    cond = (users_if.age == None) | (users_if.age > 30)
    res = users_if.get_row(
        [users_if.name.if_(cond).else_('ok')],
        order_by=users_if.id,
    )
    assert res == ['ok', 'ok', 'ok', None, 'Nima']

In [ ]:
# 5) ادامه تست‌ها — where/update/batch/delete/order_by/join/LiteralValue

# ------------------------------------------ usage in queries

def test_if_14_used_in_where(users_if):
    label = Mysql.LiteralValue('inactive').if_(users_if.active == False).else_(users_if.name)
    res = users_if.get_row(
        [users_if.id],
        where=label == 'inactive',
        order_by=users_if.id,
    )
    assert res == [3, 5]


def test_if_15_used_in_update(users_if):
    users_if.update(
        {users_if.name: Mysql.LiteralValue('unknown').if_(users_if.name == None).else_(users_if.name)},
        where=users_if.id > 0,
    )
    res = users_if.get_row([users_if.name], order_by=users_if.id)
    assert res == ['Ali', 'Reza', 'Sara', 'unknown', 'Nima']


def test_if_16_used_in_batch_update(users_if):
    batch = users_if.batch()
    batch.update(
        {users_if.name: Mysql.LiteralValue('unknown').if_(users_if.name == None).else_(users_if.name)},
        where=users_if.id > 0,
    )
    batch.run()
    res = users_if.get_row([users_if.name], order_by=users_if.id)
    assert res == ['Ali', 'Reza', 'Sara', 'unknown', 'Nima']


def test_if_17_used_in_delete(users_if):
    label = Mysql.LiteralValue('').if_(users_if.name != None).else_(users_if.name)
    users_if.delete_row(label == '')
    res = users_if.get_row([users_if.id], order_by=users_if.id)
    assert res == [4]


# ---------------------------------------------- chaining

def test_if_18_chained_string_method(users_if):
    expr = users_if.name.if_(users_if.active == True).else_('inactive').upper()
    res = users_if.get_row([expr], order_by=users_if.id)
    assert res == ['ALI', 'REZA', 'INACTIVE', None, 'INACTIVE']


def test_if_19_conditional_with_str_concat(users_if):
    expr = (
        Mysql.LiteralValue('unknown')
        .if_(users_if.name == None)
        .else_(users_if.name)
        .add_end('!')
    )
    res = users_if.get_row([expr], order_by=users_if.id)
    assert res == ['Ali!', 'Reza!', 'Sara!', 'unknown!', 'Nima!']


def test_if_20_arithmetic_on_conditional(users_if):
    expr = Mysql.LiteralValue(0).if_(users_if.age == None).else_(users_if.age) + 1
    res = users_if.get_row([expr], order_by=users_if.id)
    assert res == [31, 18, 26, 41, 1]


def test_if_21_used_in_order_by(users_if):
    label = users_if.name.if_(users_if.name == None).else_(Mysql.LiteralValue('zzz'))
    res = users_if.get_row([users_if.id], order_by=label)
    assert res == [1, 5, 2, 3, 4]


# ----------------------------------------- type / value

def test_if_22_both_branches_literal(users_if):
    res = users_if.get_row(
        [Mysql.LiteralValue('yes').if_(users_if.active == True).else_(Mysql.LiteralValue('no'))],
        order_by=users_if.id,
    )
    assert res == ['yes', 'yes', 'no', 'yes', 'no']


def test_if_23_int_branches(users_if):
    res = users_if.get_row(
        [Mysql.LiteralValue(0).if_(users_if.age == None).else_(users_if.age)],
        order_by=users_if.id,
    )
    assert res == [30, 17, 25, 40, 0]


def test_if_24_empty_string_else(users_if):
    res = users_if.get_row(
        [users_if.name.if_(users_if.name == None).else_(Mysql.LiteralValue(''))],
        order_by=users_if.id,
    )
    assert res == ['', '', '', None, '']


def test_if_25_current_datatype_is_none(users_if):
    label = users_if.name.if_(users_if.active == True).else_(Mysql.LiteralValue(0))
    assert label.current_datatype is None


def test_if_26_multiple_conditionals_in_select(users_if):
    a = users_if.name.if_(users_if.active == True).else_('X')
    b = users_if.age.if_(users_if.age == None).else_(Mysql.LiteralValue(-1))
    res = users_if.get_row([a, b], order_by=users_if.id)
    assert res == (
        ('Ali', 30),
        ('Reza', 17),
        ('X', 25),
        (None, 40),
        ('X', -1),
    )


# ------------------------------------------ joins

def test_if_27_conditional_in_join(in_driver, users_if):
    name = f"orders_if_{uuid.uuid4().hex[:8]}"
    s = Mysql.TableStructure(name)
    s.add_column("id",      Mysql.DataTypes.INT(), primary_key=True)
    s.add_column("user_id", Mysql.DataTypes.INT())
    s.add_column("total",   Mysql.DataTypes.FLOAT())
    in_driver.create_table(s)
    orders = getattr(in_driver, name)
    orders.bulk_insert(
        [orders.id, orders.user_id, orders.total],
        [(100, 1, 50.0), (101, 3, 0.0)]
    )
    expr = users_if.name.if_(orders.total == 0).else_(users_if.name)
    try:
        res = (users_if
               .inner_join(orders, users_if.id == orders.user_id)
               .get_row([expr], order_by=users_if.id))
        assert res == (('Ali',), ('Sara',))
    finally:
        try:
            in_driver.delete_table(orders, True, True, True)
        except Exception:
            pass


# -------------------------------------- LiteralValue API

def test_if_28_literal_value_exported():
    assert hasattr(Mysql, 'LiteralValue')
    lv = Mysql.LiteralValue('hello')
    assert lv._output == ('%s', ['hello'])


def test_if_29_literal_value_arithmetic():
    lv = Mysql.LiteralValue(100)
    expr = lv - 1
    assert expr._output == ('(%s - %s)', [100, 1])


def test_if_30_literal_value_string_methods():
    lv = Mysql.LiteralValue('hello').upper()
    assert lv._output == ('(UPPER(%s))', ['hello'])


def test_if_31_literal_value_in_where(users_if):
    res = users_if.get_row(
        [users_if.name],
        where=Mysql.LiteralValue(1) == users_if.id,
        order_by=users_if.id,
    )
    assert res == ['Ali']

In [ ]:
# 6) اجرای تست‌ها در همین نوتبوک
import sys
exit_code = pytest.main([
    "-v",
    "--tb=short",
    "-p", "no:cacheprovider",
    "--pyargs", "__main__",        # <-- تعریف‌شده در همین نوتبوک
], plugins=[__import__("pytest")])
print("\npytest exit code:", exit_code)
print("  0 → همه پاس ✅    1 → حداقل یک fail ❌    2/5 → خطا")

In [ ]:
# 7) جمع‌آوری تست‌ها به‌صورت دستی از globals نوتبوک و اجرا با pytest
#    (روش مطمئن‌تر برای Jupyter — چون __main__ گاهی نادیده گرفته می‌شود.)
import types
import pytest

# ساخت یک ماژول موقت به نام `notebook_tests` و انتقال همه‌ی test_if_* ها و fixture ها به آن
mod = types.ModuleType("notebook_tests")
for k, v in list(globals().items()):
    if k.startswith("test_") or k in ("in_driver", "users_if"):
        setattr(mod, k, v)

import sys
sys.modules["notebook_tests"] = mod

exit_code = pytest.main([
    "notebook_tests",
    "-k", "test_if_",
    "-v",
    "--tb=short",
    "-p", "no:cacheprovider",
])
print("\npytest exit code:", exit_code)